<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/train_e2e.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate evaluate librosa soundfile scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


In [2]:
# Cell 2: Mount Drive + paths config
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE  = '/content/drive/MyDrive/ThaiScamCall'
ZIP_PATH    = f'{DRIVE_BASE}/mp3_15s.zip'
EXTRACT_DIR = '/content/data_15s'
SPLITS_DIR  = f'{DRIVE_BASE}/splits'
RUNS_DIR    = f'{DRIVE_BASE}/runs/e2e'
os.makedirs(RUNS_DIR, exist_ok=True)

Mounted at /content/drive


In [3]:
# Cell 3: Extract zip ไปยัง local Colab
import zipfile
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) < 1000:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
print('Files:', len(os.listdir(EXTRACT_DIR)))

Files: 21287


In [4]:
# Cell 4: Build raw datasets (load audio ใน prep — librosa, ไม่ใช้ HF Audio feature)
import pandas as pd
from datasets import Dataset

def build_raw(split):
    df = pd.read_csv(f'{SPLITS_DIR}/{split}.csv')
    return Dataset.from_pandas(df[['filename', 'label']])

raw = {s: build_raw(s) for s in ['train', 'val', 'test']}
for s, d in raw.items(): print(s, len(d))

train 17029
val 2129
test 2129


In [5]:
# Cell 5: Baselines — trivial (majority/random) + MFCC classical (LR/RF)
import numpy as np, librosa
from joblib import Parallel, delayed
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

BASELINE_DIR = f'{DRIVE_BASE}/runs/baselines'
FEATURES_DIR = f'{DRIVE_BASE}/features'
os.makedirs(BASELINE_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)

splits_df = {s: pd.read_csv(f'{SPLITS_DIR}/{s}.csv') for s in ['train', 'val', 'test']}
test_df = splits_df['test']

def save_baseline_preds(name, test_df, y_pred, y_score):
    out_dir = f'{BASELINE_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame({
        'filename':  test_df.filename.values,
        'label':     test_df.label.values,
        'pred':      y_pred,
        'prob_scam': y_score,
    }).to_csv(f'{out_dir}/test_predictions.csv', index=False)
    y = test_df.label.values
    print(f'[{name}] acc={accuracy_score(y, y_pred):.4f}  '
          f'macro_f1={f1_score(y, y_pred, average="macro"):.4f}  '
          f'auc={roc_auc_score(y, y_score):.4f}')

# === Trivial ===
maj = int(splits_df['train'].label.value_counts().idxmax())
save_baseline_preds('majority', test_df,
                    np.full(len(test_df), maj),
                    np.full(len(test_df), float(maj)))
rng = np.random.RandomState(42)
save_baseline_preds('random', test_df,
                    rng.randint(0, 2, size=len(test_df)),
                    rng.uniform(size=len(test_df)))

# === MFCC features (cache ลง Drive ~10-15 นาที ครั้งแรก) ===
N_MFCC = 13
FEAT_DIM = N_MFCC * 3

def extract_mfcc(fn):
    try:
        y, sr = librosa.load(f'{EXTRACT_DIR}/{fn}', sr=16000)
        m  = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        d1 = librosa.feature.delta(m)
        d2 = librosa.feature.delta(m, order=2)
        return np.concatenate([m.mean(axis=1), d1.mean(axis=1), d2.mean(axis=1)])
    except Exception:
        return np.zeros(FEAT_DIM, dtype=np.float32)

for split, df in splits_df.items():
    cache = f'{FEATURES_DIR}/mfcc_{split}.npy'
    if os.path.exists(cache):
        print(f'{split}: cached')
        continue
    feats = Parallel(n_jobs=4)(
        delayed(extract_mfcc)(fn) for fn in tqdm(df.filename, desc=f'mfcc/{split}')
    )
    np.save(cache, np.array(feats, dtype=np.float32))

X_train = np.load(f'{FEATURES_DIR}/mfcc_train.npy')
X_test  = np.load(f'{FEATURES_DIR}/mfcc_test.npy')
y_train = splits_df['train'].label.values
y_test  = splits_df['test'].label.values

# === MFCC + LR ===
sc = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1).fit(sc.transform(X_train), y_train)
save_baseline_preds('mfcc_lr', test_df,
                    lr.predict(sc.transform(X_test)),
                    lr.predict_proba(sc.transform(X_test))[:, 1])

# === MFCC + RF ===
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42).fit(X_train, y_train)
save_baseline_preds('mfcc_rf', test_df,
                    rf.predict(X_test),
                    rf.predict_proba(X_test)[:, 1])

[majority] acc=0.5242  macro_f1=0.3439  auc=0.5000
[random] acc=0.5087  macro_f1=0.5081  auc=0.4827
train: cached
val: cached
test: cached
[mfcc_lr] acc=0.6322  macro_f1=0.6300  auc=0.6820
[mfcc_rf] acc=0.6628  macro_f1=0.6599  auc=0.7366


In [6]:
# Cell 6: Backbones config (3 ตัวแทน paradigm — ตัดลงเพื่อประหยัด compute units)
BACKBONES = {
    'xlsr':        'facebook/wav2vec2-xls-r-300m',   # wav2vec2-XLS-R
    'wavlm':       'microsoft/wavlm-base-plus',       # WavLM (masked + denoising)
    'whisper_enc': 'openai/whisper-small',            # Whisper encoder
}
# ถ้ามี budget เพิ่ม เปิดบรรทัดนี้ได้:
# BACKBONES['xlsr53']       = 'facebook/wav2vec2-large-xlsr-53'
# BACKBONES['hubert']       = 'facebook/hubert-base-ls960'
# BACKBONES['whisper_thai'] = 'biodatlab/whisper-th-small-combined'

In [7]:
# Cell 7: Fine-tune end-to-end (loop backbones, Whisper-aware padding)
import torch, gc, numpy as np, evaluate, librosa, warnings
warnings.filterwarnings('ignore')
from transformers import (AutoFeatureExtractor, AutoModelForAudioClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
f1_m  = evaluate.load('f1')
acc_m = evaluate.load('accuracy')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        **acc_m.compute(predictions=preds, references=p.label_ids),
        **f1_m.compute(predictions=preds, references=p.label_ids, average='macro'),
    }
SR = 16000
MAX_LEN = SR * 15        # 15s for wav2vec2/wavlm
BATCH_SIZE = {
    'xlsr':        16,
    'wavlm':       32,
    'whisper_enc': 16,
}
def train_backbone(name, model_id):
    out_dir = f'{RUNS_DIR}/{name}'
    os.makedirs(out_dir, exist_ok=True)
    print(f'\n========== {name} ({model_id}) ==========')
    fe = AutoFeatureExtractor.from_pretrained(model_id)
    is_whisper = 'whisper' in model_id.lower()
    def prep(batch):
        audios = []
        for fn in batch['filename']:
            try:
                audio, _ = librosa.load(f'{EXTRACT_DIR}/{fn}', sr=SR)
            except Exception:
                audio = np.zeros(SR, dtype=np.float32)
            audios.append(audio)
        # Whisper FE auto-pads to 30s (n_samples=480000); wav2vec2/wavlm ใช้ MAX_LEN
        if is_whisper:
            out = fe(audios, sampling_rate=SR)
        else:
            out = fe(audios, sampling_rate=SR, max_length=MAX_LEN,
                     truncation=True, padding='max_length')
        out['label'] = batch['label']
        return out
    ds = {s: raw[s].map(prep, batched=True, batch_size=32,
                        remove_columns=[c for c in raw[s].column_names
                                        if c not in ['filename', 'label']])
          for s in ['train', 'val', 'test']}
    model = AutoModelForAudioClassification.from_pretrained(model_id, num_labels=2)
    if hasattr(model, 'freeze_feature_encoder'):
        model.freeze_feature_encoder()
    bs = BATCH_SIZE.get(name, 8)
    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=5,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=max(1, 32 // bs),
        per_device_eval_batch_size=bs * 2,
        learning_rate=3e-5,
        warmup_steps=300,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        fp16=True,
        logging_steps=50,
        save_total_limit=1,
        report_to='none',
        dataloader_num_workers=4,
    )
    trainer = Trainer(
        model=model, args=args,
        train_dataset=ds['train'].remove_columns(['filename']),
        eval_dataset=ds['val'].remove_columns(['filename']),
        processing_class=fe,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    test_no_fn = ds['test'].remove_columns(['filename'])
    pred = trainer.predict(test_no_fn)
    y_pred  = np.argmax(pred.predictions, axis=1)
    y_true  = pred.label_ids
    y_score = torch.softmax(torch.tensor(pred.predictions), dim=1)[:, 1].numpy()
    test_df = pd.read_csv(f'{SPLITS_DIR}/test.csv')
    out = pd.DataFrame({
        'filename':  test_df.filename.values,
        'label':     y_true,
        'pred':      y_pred,
        'prob_scam': y_score,
    })
    pred_path = f'{out_dir}/test_predictions.csv'
    out.to_csv(pred_path, index=False)
    print(classification_report(y_true, y_pred, target_names=['not_scam', 'scam'], digits=4))
    print('Confusion:\n', confusion_matrix(y_true, y_pred))
    print('ROC-AUC:', roc_auc_score(y_true, y_score))
    print('saved ->', pred_path)
    del trainer, model, fe, ds
    gc.collect()
    torch.cuda.empty_cache()
for name, mid in BACKBONES.items():
    train_backbone(name, mid)


========== xlsr (facebook/wav2vec2-xls-r-300m) ==========


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

Map:   0%|          | 0/17029 [00:00<?, ? examples/s]

Map:   0%|          | 0/2129 [00:00<?, ? examples/s]

Map:   0%|          | 0/2129 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    not_scam     0.9847    0.9830    0.9839      1116
        scam     0.9813    0.9832    0.9822      1013

    accuracy                         0.9831      2129
   macro avg     0.9830    0.9831    0.9831      2129
weighted avg     0.9831    0.9831    0.9831      2129

Confusion:
 [[1097   19]
 [  17  996]]
ROC-AUC: 0.9988036351799368
saved -> /content/drive/MyDrive/ThaiScamCall/runs/e2e/wavlm/test_predictions.csv
